In [1]:
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image
from crossword_algorithm import find_words
import random

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/opt/anaconda3/envs/testenv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

20 words found by find_words
[((0, 0), (0, 5)), ((17, 4), (17, 9)), ((0, 14), (7, 14)), ((17, 14), (14, 11)), ((17, 0), (14, 3)), ((5, 4), (5, 7)), ((4, 14), (4, 11)), ((5, 7), (8, 7)), ((14, 9), (10, 9)), ((9, 2), (9, 5)), ((8, 12), (13, 12)), ((7, 0), (10, 0)), ((2, 6), (2, 1)), ((4, 0), (4, 7)), ((16, 11), (16, 4)), ((12, 0), (12, 5)), ((0, 11), (5, 11)), ((11, 14), (15, 14)), ((3, 11), (3, 4)), ((12, 5), (7, 10))]


In [2]:
image2 = cv2.imread("image_samples/word_search_school.png")
image = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)
#ret, image = cv2.threshold(image, 100, 255, cv2.THRESH_BINARY_INV)
image = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 33, 25)
image = cv2.morphologyEx(image, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

#image = cv2.blur(image, (5,5))

contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

area = 0

for cnt in contours:
    area += cv2.contourArea(cnt)

area = area/len(contours)

letter_grid_coordinates = dict()
y_top, y_bottom = -1,-1

for i in range(len(contours)):
    #cv2.drawContours(image2, contours[i], -1, (255,0,0), 7)
    if cv2.contourArea(contours[i]) < 0.1*area:
        continue
    x, y, w, h = cv2.boundingRect(contours[i])
    #cv2.rectangle(image2, (x - int(w*(0.45)), y - int(h*(0.45))), (x + w + int(w*(0.45)), y + h + int(h*(0.45))), (0, 255, 0), 2)

    pushed = False

    for key in letter_grid_coordinates.keys():
        if y + w//2 in range(key[0], key[1]):
            y_top, y_bottom = y, y+h
            letter_grid_coordinates[key].append((x - int((w)*(0.25)), y - int(h*(0.25)), x + w + int(w*(0.25)), y + h + int(h*(0.25))))
            pushed = True

    if not pushed:
        letter_grid_coordinates[(y, y+w)] = [(x - int(w*(0.25)), y - int(h*(0.25)), x + w + int(w*(0.25)), y + h + int(h*(0.25)))]
    
    #cv2.rectangle(image2, (x - int(w*(0.45)), y - int(h*(0.45))), (x + w + int(w*(0.45)), y + h + int(h*(0.45))), (0, 255, 0), 2)
    

In [3]:
model = tf.keras.models.load_model("font_identifier.keras")
letter_width = None

# for x in letter_grid_coordinates.keys():
#     print(len(letter_grid_coordinates[x]))

In [4]:
cv2.imshow("unsolved", image2)
#cv2.imshow("image", image)
cv2.waitKey(1000)
cv2.destroyAllWindows()

crossword = list()

for i in letter_grid_coordinates.keys():
    letter_grid_coordinates[i] = sorted(letter_grid_coordinates[i], key=lambda x: x[0])
    #print(letter_grid_coordinates[i])

all_images = list()

for i in reversed(letter_grid_coordinates.values()):
    for j in i:
        x1, y1, x2, y2 = j
        if not letter_width:
            letter_width = y2 - y1
        img = image[y1:y2, x1:x2]
        img = Image.fromarray(img)
        img.thumbnail((28, 28), Image.Resampling.LANCZOS)

        pillow_image = Image.new("L", (28, 28), 0)
        pillow_image.paste(img, ((28 - img.size[0]) // 2, (28 - img.size[1]) // 2))

        img = np.array(pillow_image).reshape((28,28,1))
        img = img/255.0

        all_images.append(img)

all_images = np.array(all_images)
predictions = model.predict(all_images, len(all_images))

output = list()

for prediction in predictions:
    output.append(chr(np.argmax(prediction) + 65))

output = np.array(output)
output = output.reshape((len(letter_grid_coordinates.values()), len(list(letter_grid_coordinates.values())[0])))
output = output.tolist()

crossword = output
for row in crossword:
    for letter in row:
        print(letter, end=" ")
    print("")

print(output)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
X G O W P Z D R A M A F K Y R A R B I L 
R N S N S T C E J B U S H E Q M A T H V 
L I T A O F Y S E N X P K A I C W G D Z 
M T U K E I B H V L A Q N O L U R J T S 
Y I D H C W T X T R E Z F A O S E N E P 
J R E Q O A N A G L U M S K V B A I B Y 
S W N X S M P O C H A S E T A Z D R A F 
C E T O T Y E K B U R E V N J U I L H Q 
I Z S F R G A W C O D L H E T L N X P M 
E Q J S A T V R O A K E P S Y A G O L E 
N U M B E R S M S R B T L I D F R S A H 
C R X V G C O J H C K A W A Z Q P Y N O 
E O H P A F E N G L I S H O C E B U T D 
B C I S U M K R F C Y S L X L I C N E P 
A J S R G V Q B O Z M O S L W T S H A U 
P X T U N D E S K F O B I O S R M Y C N 
S N O Y A R C Z W H J N T V R A Q O H L 
U F R I L B M H C T G R A D E S U K E P 
H M Y T O C G S N R A E L S D N E I R F 
[['X', 'G', 'O', 'W', 'P', 'Z', 'D', 'R', 'A', 'M', 'A', 'F', 'K', 'Y', 'R', 'A', 'R', 'B', 'I', 'L'], ['R', 'N', 'S', 'N', 'S', 'T', 'C', 'E', 'J', 'B', 'U', 'S', 'H', 'E', 'Q', 'M',

In [5]:
# word_position = find_words(crossword, ['GARDEN', 'SUMMER', 'SUNSHINE', 'SWIM', 'BOAT', 'CAMP', 'HIKE', 'PLAY', 'BEACH', 'JULY', 'AUGUST', 'PARK', 'PICNIC', 'POPSICLE', 'ICECREAM', 'SHORTS', 'TRAVEL', 'DRESS', 'VACATION', 'SEASON'])

word_position = find_words(crossword, [
    'DRAMA', 'HISTORY', 'NUMBERS', 'SCIENCE', 'ART',
    'ELEMENTARY', 'HOMEWORK', 'PENCIL', 'SOCIALSTUDIES',
    'BACKPACK', 'ENGLISH', 'LANGUAGEARTS', 'PHYSICALEDUCATION',
    'SPELLING', 'BOOKS', 'FRIENDS', 'LEARN', 'READING',
    'STUDENTS', 'CLASSROOM', 'GEOGRAPHY', 'LIBRARY',
    'RECESS', 'SUBJECTS', 'CRAYONS', 'GRADES',
    'MATH', 'SCHOOL', 'TEACHER', 'DESK', 'HEALTH',
    'MUSIC', 'SCISSORS', 'WRITING'
])

image2_cpy = image2.copy()

for x in word_position:
    start_word_pos, end_word_pos = x
    k = list(reversed(letter_grid_coordinates.values()))
    start = k[start_word_pos[0]][start_word_pos[1]]
    end = k[end_word_pos[0]][end_word_pos[1]]

    color = random.sample(range(30, 200), 3)

    #result = cv2.line(result, ((start[0] + start[2]) // 2 - 10, (start[1] + start[3]) // 2 - 10) , ((end[0] + end[2]) // 2 + 10, (end[1] + end[3]) // 2 + 10) , color, 1)
    result = cv2.line(image2, ((start[0] + start[2]) // 2 - 2, (start[1] + start[3]) // 2 - 2) , ((end[0] + end[2]) // 2 + 2, (end[1] + end[3]) // 2 + 2) , color, letter_width*10//14)
    
alpha = 0.45
result = cv2.addWeighted(result, alpha, image2_cpy, 1 - alpha, 0)

cv2.imshow("solved", result)
cv2.waitKey(10)


34 words found by find_words


-1